In [3]:
# ===========================
# Install Dependencies
# ===========================

!pip install -q tensorflow keras transformers streamlit nltk sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 57.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 80.7 MB/s eta 0:00:00:00:0100:01


In [4]:
# ==========================================
# Import Libraries
# ==========================================

import os
import re
import json
import pickle
from datetime import datetime

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from typing import Dict, Any

# Download NLTK resources
nltk.download("punkt")
nltk.download("stopwords")

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

from transformers import AutoTokenizer, BertForSequenceClassification
from tensorflow.keras.models import load_model
from datetime import datetime

In [6]:
BILSTM_MODEL_PATH = "/kaggle/input/models/harshitmehan/bilstm/keras/default/1/bilstm_emotion_model.keras"

bilstm_model = load_model(BILSTM_MODEL_PATH)

print("BiLSTM Model Loaded Successfully")

I0000 00:00:1783131345.603463      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783131345.606696      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


BiLSTM Model Loaded Successfully


In [7]:
#BERT MODEL LOADING
from transformers import AutoTokenizer, BertForSequenceClassification
import torch

BERT_MODEL_PATH = "/kaggle/input/models/harshitmehan/bert/pytorch/default/1"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_PATH)

bert_model = BertForSequenceClassification.from_pretrained(BERT_MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model.to(device)

bert_model.eval()

print("✅ BERT Model Loaded Successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ BERT Model Loaded Successfully


In [8]:
#EMOTION LABELS
emotion_labels = [
    "Bored",
    "Confident",
    "Confused",
    "Curious",
    "Frustrated"
]

print("Emotion Labels Loaded")
print(emotion_labels)

Emotion Labels Loaded
['Bored', 'Confident', 'Confused', 'Curious', 'Frustrated']


In [9]:
#TEXT PREPROCESSING AND CLEANING
import re

def clean_text(text):

    text = str(text).lower()

    # Keep punctuation that can indicate emotion
    text = re.sub(r"[^a-zA-Z0-9\s!?.,]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

print("Text Cleaning Function Ready")

Text Cleaning Function Ready


In [10]:
#KEYWORD ENHANCEMENT DICTIONARY
emotion_keywords = {

    "Frustrated": [
        "frustrated","frustrating","angry","annoying",
        "hate","stuck","difficult","wrong","problem"
    ],

    "Curious": [
        "why","how","what","learn","explore",
        "wonder","interesting","want to know"
    ],

    "Confident": [
        "easy","clear","understand","understood",
        "got it","perfect","excellent","confident"
    ],

    "Bored": [
        "boring","bored","dull","repetitive",
        "sleepy","not engaging"
    ],

    "Confused": [
        "confused","unclear","lost",
        "don't understand","doesn't make sense",
        "unsure","puzzled"
    ]
}

print("Keyword Dictionary Ready")

Keyword Dictionary Ready


In [11]:
#KEYWORD SCORING
def keyword_scores(text):

    scores = {emotion: 0 for emotion in emotion_labels}

    text = text.lower()

    for emotion, keywords in emotion_keywords.items():

        for keyword in keywords:

            if keyword in text:

                if keyword in [
                    "frustrated",
                    "confused",
                    "confident",
                    "curious",
                    "boring"
                ]:
                    scores[emotion] += 10
                else:
                    scores[emotion] += 1

    return scores

print("Keyword Scoring Ready")

Keyword Scoring Ready


In [12]:
#PROBABILITY BOOSTING
import numpy as np

def apply_keyword_boost(probabilities, keyword_score):

    probs = np.array(probabilities).copy()

    for i, emotion in enumerate(emotion_labels):

        probs[i] += keyword_score[emotion] * 0.02

    probs = probs / np.sum(probs)

    return probs

print("Probability Boost Ready")

Probability Boost Ready


In [13]:
#BILSTM PREDICTION
def predict_bilstm(text):

    cleaned = clean_text(text)

    try:

        # Expected tokenizer (if available later)
        sequence = tokenizer.texts_to_sequences([cleaned])

        sequence = pad_sequences(
            sequence,
            maxlen=80,
            padding="post"
        )

        probs = bilstm_model.predict(
            sequence,
            verbose=0
        )[0]

        probs = apply_keyword_boost(
            probs,
            keyword_scores(cleaned)
        )

        emotion = emotion_labels[np.argmax(probs)]

        return {

            "model":"BiLSTM",

            "emotion":emotion,

            "confidence":float(np.max(probs)),

            "scores":{
                emotion_labels[i]:float(probs[i])
                for i in range(len(emotion_labels))
            },

            "cleaned_text":cleaned

        }

    except Exception:

        return {

            "model":"BiLSTM",

            "emotion":"Unavailable",

            "confidence":0.0,

            "scores":{},

            "cleaned_text":cleaned

        }

print("BiLSTM Prediction Ready")

BiLSTM Prediction Ready


In [14]:
#BERT CLASS WEIGHTS
class_weights = np.array([
    1.2,    # Bored
    1.8,    # Confident
    0.6,    # Confused
    1.0,    # Curious
    1.4     # Frustrated
])

print("Class Weights Loaded")

Class Weights Loaded


In [19]:
def predict_bert(text):

    cleaned = clean_text(text)

    inputs = bert_tokenizer(
        cleaned,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    bert_label = int(np.argmax(probs))
    bert_confidence = float(np.max(probs))

    return {
        "model": "BERT",
        "bert_label": f"LABEL_{bert_label}",
        "bert_confidence": bert_confidence,
        "raw_scores": probs.tolist(),
        "cleaned_text": cleaned
    }

print("✅ BERT Prediction Ready")

✅ BERT Prediction Ready


In [20]:
def internship_emotion(text):

    cleaned = clean_text(text)

    scores = keyword_scores(cleaned)

    if max(scores.values()) == 0:
        emotion = "Curious"
        confidence = 0.50
    else:
        emotion = max(scores, key=scores.get)
        confidence = min(scores[emotion] / 10, 1.0)

    return {
        "emotion": emotion,
        "confidence": confidence,
        "scores": scores
    }

print("✅ Internship Emotion Ready")

✅ Internship Emotion Ready


In [21]:
#MIXED EMOTION DETECTION
def detect_mixed_emotions(scores, threshold=0.15):

    total = sum(scores.values())

    if total == 0:
        return []

    mixed = []

    for emotion, score in scores.items():

        percentage = score / total

        if percentage >= threshold:
            mixed.append((emotion, round(percentage, 3)))

    mixed.sort(key=lambda x: x[1], reverse=True)

    return mixed

print("✅ Mixed Emotion Detection Ready")

✅ Mixed Emotion Detection Ready


In [22]:
#UNIFIED PREDICTION SCHEMA
def predict(text):

    bert = predict_bert(text)

    internship = internship_emotion(text)

    mixed = detect_mixed_emotions(
        internship["scores"]
    )

    return {

        "input_text": text,

        "cleaned_text": bert["cleaned_text"],

        "bert_prediction": bert,

        "final_emotion": internship["emotion"],

        "confidence": internship["confidence"],

        "mixed_emotions": mixed,

        "timestamp": str(datetime.now())

    }

print("✅ Unified Prediction Schema Ready")

✅ Unified Prediction Schema Ready


In [23]:
#TEST
result = predict(
    "I am confused but curious to understand machine learning."
)

result

{'input_text': 'I am confused but curious to understand machine learning.',
 'cleaned_text': 'i am confused but curious to understand machine learning.',
 'bert_prediction': {'model': 'BERT',
  'bert_label': 'LABEL_6',
  'bert_confidence': 0.5558068156242371,
  'raw_scores': [0.002358788624405861,
   0.0019580982625484467,
   0.0029526783619076014,
   0.005664575845003128,
   0.0030364389531314373,
   0.0008709708345122635,
   0.5558068156242371,
   0.31932327151298523,
   0.00128311722073704,
   0.005979117006063461,
   0.005701242480427027,
   0.0017948935274034739,
   0.0041239503771066666,
   0.0029908232390880585,
   0.0028255849611014128,
   0.00073837029049173,
   0.0018858793191611767,
   0.0005320376949384809,
   0.0007873181602917612,
   0.0017908337758854032,
   0.0011241480242460966,
   0.0017297114245593548,
   0.00939268060028553,
   0.0008517958340235054,
   0.0023102134000509977,
   0.0031370637007057667,
   0.036029018461704254,
   0.02302057296037674],
  'cleaned_text

In [25]:
#EMOTION RESPONSE MAPPING
emotion_response = {

    "Confused":
        "It seems you're confused. Let's break the concept into smaller parts.",

    "Curious":
        "Great curiosity! Keep exploring and asking questions.",

    "Frustrated":
        "Don't worry. Take a short break and try solving one step at a time.",

    "Confident":
        "Excellent! You seem confident about this topic. Keep practicing.",

    "Bored":
        "Let's make learning more engaging with examples and exercises."

}

print("Response Mapping Ready")

Response Mapping Ready


In [28]:
#ATTACH RESPONSE
def generate_response(text):

    result = predict(text)

    result["response"] = emotion_response[
        result["final_emotion"]
    ]

    return result

print("Response Generator Ready")

Response Generator Ready


In [30]:
#CSV PERSISTENCE
import os

CSV_FILE = "emotion_predictions.csv"

def save_prediction(result):

    row = {

        "Timestamp": result["timestamp"],

        "Input": result["input_text"],

        "Emotion": result["final_emotion"],

        "Confidence": result["confidence"],

        "Response": result["response"]

    }

    df = pd.DataFrame([row])

    if os.path.exists(CSV_FILE):

        df.to_csv(

            CSV_FILE,

            mode="a",

            header=False,

            index=False

        )

    else:

        df.to_csv(

            CSV_FILE,

            index=False

        )

print("CSV Saving Ready")

CSV Saving Ready


In [31]:
#TEST SAVING
result = generate_response(
    "I am really frustrated because I cannot understand neural networks."
)

save_prediction(result)

result

{'input_text': 'I am really frustrated because I cannot understand neural networks.',
 'cleaned_text': 'i am really frustrated because i cannot understand neural networks.',
 'bert_prediction': {'model': 'BERT',
  'bert_label': 'LABEL_3',
  'bert_confidence': 0.34419581294059753,
  'raw_scores': [0.0023641998413950205,
   0.004139531869441271,
   0.22658392786979675,
   0.34419581294059753,
   0.005696977488696575,
   0.0038834712468087673,
   0.019304892048239708,
   0.01580522209405899,
   0.0018893901724368334,
   0.15624623000621796,
   0.051572609692811966,
   0.028018061071634293,
   0.012314319610595703,
   0.0016215145587921143,
   0.0064722695387899876,
   0.0010718507692217827,
   0.004442879930138588,
   0.0016474322183057666,
   0.0021750377491116524,
   0.00516105629503727,
   0.004797409288585186,
   0.0038176579400897026,
   0.008328474126756191,
   0.0019556942861527205,
   0.004776983521878719,
   0.05130757763981819,
   0.006849419325590134,
   0.02356010489165783],
 

In [32]:
history = pd.read_csv("emotion_predictions.csv")

history.tail()

,Timestamp,Input,Emotion,Confidence,Response
0,2026-07-04 02:29:52.487732,I am really frustrated because I cannot unders...,Frustrated,1.0,Don't worry. Take a short break and try solvin...


In [33]:
#CACHED MODEL LOADING(STREAMLIT READY)
try:

    import streamlit as st

    @st.cache_resource
    def load_models():

        return {

            "bert_model": bert_model,

            "bert_tokenizer": bert_tokenizer,

            "bilstm_model": bilstm_model

        }

    print("Streamlit Cache Ready")

except:

    print("Running inside Notebook")

Streamlit Cache Ready


In [34]:
text = input("Enter your sentence: ")

result = generate_response(text)

print("=" * 50)

print("Detected Emotion :", result["final_emotion"])

print("Confidence :", round(result["confidence"], 3))

print("Response :")

print(result["response"])

print("=" * 50)

save_prediction(result)

Enter your sentence:  MY NAME IS HARSHIT MEHAN


Detected Emotion : Curious
Confidence : 0.5
Response :
Great curiosity! Keep exploring and asking questions.


In [35]:
examples = [

    "I don't understand anything.",

    "This topic is very interesting.",

    "I am frustrated with coding.",

    "Everything is clear now.",

    "This lecture is boring."

]

for sentence in examples:

    print("\nSentence :", sentence)

    output = generate_response(sentence)

    print("Emotion :", output["final_emotion"])

    print("Confidence :", round(output["confidence"],3))

    print("Response :", output["response"])

    print("-"*60)


Sentence : I don't understand anything.
Emotion : Confident
Confidence : 0.1
Response : Excellent! You seem confident about this topic. Keep practicing.
------------------------------------------------------------

Sentence : This topic is very interesting.
Emotion : Curious
Confidence : 0.1
Response : Great curiosity! Keep exploring and asking questions.
------------------------------------------------------------

Sentence : I am frustrated with coding.
Emotion : Frustrated
Confidence : 1.0
Response : Don't worry. Take a short break and try solving one step at a time.
------------------------------------------------------------

Sentence : Everything is clear now.
Emotion : Confident
Confidence : 0.1
Response : Excellent! You seem confident about this topic. Keep practicing.
------------------------------------------------------------

Sentence : This lecture is boring.
Emotion : Bored
Confidence : 1.0
Response : Let's make learning more engaging with examples and exercises.
-------